In [1]:
import evaluate
import pandas as pd

rouge = evaluate.load('rouge')

/home/adeluci2/.conda/envs/mds/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load predictions and output
target_path = "/home/adeluci2/projects/multidoc_summarization/data/convosumm/convosumm-data-original/data-processed/reddit/arg-filtered/test.target"
output_path = "/home/adeluci2/projects/multidoc_summarization/data/convosumm/convosumm-data-original/data-processed/reddit/output/reddit.arg-filtered.txt"

data = []
with open(target_path) as f_target:
    with open(output_path) as f_output:
        for line_t, line_o in zip(f_target, f_output):
            row = {
                "reference": line_t.strip(),
                "predictions": line_o.strip()
            }
            data.append(row)

df = pd.DataFrame(data)

# ROUGE

In [3]:
results = rouge.compute(
    predictions=df["predictions"],
    references=df["reference"],
    use_aggregator=False
)

for k,v in results.items():
    df[k] = v

In [5]:
df.mean(numeric_only=True)

rouge1       0.334203
rouge2       0.098091
rougeL       0.205473
rougeLsum    0.205473
dtype: float64

# MoverScore

In [6]:
import sys
import os
sys.path.append(f"{os.environ['MDS']}/code")
from evaluate_generations import eval_moverscore

[nltk_data] Downloading package punkt to
[nltk_data]     /home/adeluci2/scratch4-mdredze1/nltk_cache...
[nltk_data]   Package punkt is already up-to-date!


In [7]:
results = eval_moverscore(
    df.to_dict(orient="list"),
    device="cpu"
)

/home/adeluci2/.conda/envs/mds/lib/python3.12/site-packages/pytorch_pretrained_bert/modeling.py:603: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(we

In [9]:
df["moverscore"] = results["individual"]

# Analyze

In [11]:
for i, row in df.sort_values(by="moverscore", ascending=False).iterrows():
    print(f"""{i} Rouge1: {row.rouge1:.2} RougeLSumm: {row.rougeLsum:.2} MoverScore: {row.moverscore:.2}
    Generated: {row.predictions}
    
    Target: {row.reference}
    --------------------
    """)

242 Rouge1: 0.5 RougeLSumm: 0.36 MoverScore: 0.3
    Generated: Commenters discuss what it would be like to see sunlight for the first time on Earth. A few commenters say that it would look like a sunrise, while another commenter says that it'd look like the sunrise of another planet. Another commenter states that there would be no life because there's no sunlight on the planet. One user says that they would all go blind and die from heat exhaustion if they didn't have sunlight.
    
    Target: All commenters discuss what it would be like if the earth didn't have a sun and then suddenly did. A few commenters point out that everything would be different since every kind of life on the planet and pretty much everything else has been affected by having a sun since forever. A few commenters mention how bright it would seem since without a sun there wouldn't be any moonlight either. One commenter says that all humans would go blind and then die from heat exhaustion.
    -------------------